# PrecisionSkin - Export to YOLO, Train, Evaluate  (lesion detector)

Takes the **audited** data (from `PrecisionSkin_Localization_Dataset_Audit`) and does everything else in one place:

1. read the audit reports (`master_metadata.csv`, `annotations_long.csv`, `all_issues.csv`) - your original folders are only **read**
2. filter (decisions, issues, labels), then make a **leak-free split by `duplicate_group`**
3. write a YOLO dataset + a manifest that records the origin of every box
4. train a detector, evaluate it on the held-out **test** split, benchmark it on CPU, export ONNX

**Where do the boxes come from?** From the human annotators of the source datasets (Roboflow projects). Nothing is machine-drawn in this notebook, and
nobody has to draw boxes for the test set: the test images are photos whose boxes were drawn by people, and **their whole duplicate groups are kept out of training**.

**Split rules (why the scores will be honest)**
- All copies of one photo (`duplicate_group`) always land in the same split.
- Validation and test contain **facial, kept** images only, **one image per photo** (the highest-resolution copy), so augmented copies do not inflate the score.
- Photos that appear under two different conditions are excluded from everything (their label is in doubt).
- Training may also use images still in your "review" pile (optional), never for evaluation.

Run the cells in order. Training needs a GPU and `ultralytics` (see the install cell).

In [ ]:
# 0. install (run once). Use the venv you train with; PyTorch with CUDA must already be installed there.
%pip install -q ultralytics onnx onnxruntime pyyaml opencv-python-headless matplotlib pandas

## Stage 1 - Config  (edit this cell only)

In [ ]:
import os, json, random, shutil, collections, hashlib, datetime, time
from pathlib import Path
import numpy as np, pandas as pd, cv2, yaml
import matplotlib.pyplot as plt

# --- inputs: the audit notebook's output folder -----------------------------------------------------------------
OUT_ROOT = Path.home() / "PrecisionSkinV1" / "localization_dataset"
AUDIT = OUT_ROOT / "01_audit_reports"

# --- output: the YOLO dataset and the training runs go here -----------------------------------------------------
WORK = Path.home() / "PrecisionSkinV1" / "detector_work"
DATASET_VERSION = "v1"                      # bump when you rebuild with different rules, so results stay reproducible

CLASSES = ["acne", "hyperpigmentation", "eczema"]
COND_TO_CLASS = {"Acne": "acne", "Hyperpigmentation": "hyperpigmentation", "Eczema": "eczema"}

# --- what may be used -----------------------------------------------------------------------------------------
TRAIN_DECISIONS = ["keep", "review"]        # images still in your review pile may TRAIN ("keep" only = stricter)
EVAL_DECISIONS = ["keep"]                   # validation/test: reviewed-clean images only
EXCLUDE_ISSUES = ["zero_area_box", "corrupted_image", "mixed_conditions", "duplicate_across_conditions", "low_resolution"]
EVAL_FACIAL_ONLY = True                     # validation/test contain facial images only (your system analyses faces)
TRAIN_NON_FACIAL = True                     # non-facial images (if any) may train, never evaluate
MIN_BOX_PIXELS = 3                          # drop boxes thinner than this (in the original image)
KEEP_EMPTY_FRACTION = 0.03                  # share of no-box images kept as background examples

# --- split ----------------------------------------------------------------------------------------------------
SEED = 42
EVAL_FRACTION = 0.10                        # of eligible PHOTOS per class, for validation and again for test
MIN_EVAL_PHOTOS = 30                        # at least this many photos per class in val and in test (small classes!)
MAX_EVAL_FRACTION = 0.25                    # ...but never more than this share of a class
MAX_COPIES_PER_GROUP = 3                    # training: at most this many copies (augmentations) of one photo
MAX_TRAIN_PHOTOS_PER_CLASS = None           # e.g. 1500 to balance classes (None = no cap)
OVERSAMPLE = {"eczema": 3}                  # class -> total times its TRAINING images are used (weakest class). {} = off

# --- training ---------------------------------------------------------------------------------------------------
MODEL = "yolov8n.pt"                        # n = fastest on the Mini PC CPU. Try yolov8s.pt if accuracy is too low and latency allows
IMGSZ = 640
EPOCHS = 100
PATIENCE = 25
CONF = 0.25
SEEDS = [42]                                # e.g. [42, 7, 123] to report the spread for the small eczema class (3x training time)
WORKERS = 2                                 # data-loading processes. Each one needs RAM; with ~7 GB of RAM keep this at 1-2 (0 = slowest, safest)
BATCH = 16                                  # fixed batch size (auto-batch can pick too much for a shared-memory laptop)

ds_dir = WORK / ("dataset_" + DATASET_VERSION)
assert (AUDIT / "master_metadata.csv").exists(), "Run the audit notebook first: %s not found" % AUDIT
print("audit reports :", AUDIT)
print("dataset dir   :", ds_dir)

## Stage 2 - Load the audit and filter

In [ ]:
master = pd.read_csv(AUDIT / "master_metadata.csv", low_memory=False)
ann = pd.read_csv(AUDIT / "annotations_long.csv", low_memory=False)
issues = pd.read_csv(AUDIT / "all_issues.csv", low_memory=False)
for c in ("cx", "cy", "w", "h"):
    ann[c] = pd.to_numeric(ann[c], errors="coerce")
ann = ann.dropna(subset=["cx", "cy", "w", "h"])
print("master images:", len(master), "| annotations:", len(ann))

# --- images that carry an excluded issue are removed everywhere ---
bad_ids = set(issues[issues.issue.isin(EXCLUDE_ISSUES)].image_id)
m = master[~master.image_id.isin(bad_ids)].copy()
print("removed by EXCLUDE_ISSUES:", len(master) - len(m))

m = m[m.decision.isin(set(TRAIN_DECISIONS) | set(EVAL_DECISIONS))]
m = m[m.condition.isin(COND_TO_CLASS)]
m = m[~m.corrupt.astype(bool)]
print("after decision/condition filter:", len(m))

# --- boxes: only labels that map to a detector class, big enough, clipped to the image ---
ann = ann[ann.image_id.isin(set(m.image_id))].copy()
ann = ann[ann.standardized_label.isin(COND_TO_CLASS)]
ann["cls"] = ann.standardized_label.map(COND_TO_CLASS)
dims = m.set_index("image_id")[["width", "height"]]
ann = ann.join(dims, on="image_id")
ann["x0"] = (ann.cx - ann.w / 2).clip(0, 1); ann["x1"] = (ann.cx + ann.w / 2).clip(0, 1)
ann["y0"] = (ann.cy - ann.h / 2).clip(0, 1); ann["y1"] = (ann.cy + ann.h / 2).clip(0, 1)
ok_box = ((ann.x1 - ann.x0) * ann.width >= MIN_BOX_PIXELS) & ((ann.y1 - ann.y0) * ann.height >= MIN_BOX_PIXELS)
print("boxes dropped (too small / degenerate):", int((~ok_box).sum()), "of", len(ann))
ann = ann[ok_box]
ann["ncx"] = (ann.x0 + ann.x1) / 2; ann["ncy"] = (ann.y0 + ann.y1) / 2; ann["nw"] = ann.x1 - ann.x0; ann["nh"] = ann.y1 - ann.y0

nbox = ann.groupby("image_id").size()
m["n_kept_boxes"] = m.image_id.map(nbox).fillna(0).astype(int)
withbox = m[m.n_kept_boxes > 0]
empty = m[m.n_kept_boxes == 0].sample(frac=1.0, random_state=SEED)
empty = empty.iloc[: int(len(withbox) * KEEP_EMPTY_FRACTION)]
m = pd.concat([withbox, empty])
print("images kept:", len(m), "(with boxes %d, background %d)" % (len(withbox), len(empty)))
print(m.groupby(["condition", "decision"]).size().unstack(fill_value=0).to_string())

## Stage 3 - Leak-free split by duplicate group

In [ ]:
rng = random.Random(SEED)
m["pixels"] = m.width * m.height
g = m.groupby("duplicate_group")
ginfo = pd.DataFrame({
    "n": g.size(),
    "conds": g.condition.agg(lambda s: set(s)),
    "all_facial": g.facial_status.agg(lambda s: bool((s == "facial").all())),
    "any_eval_ok": g.decision.agg(lambda s: bool(s.isin(EVAL_DECISIONS).any())),
})
ginfo["single_cond"] = ginfo.conds.map(lambda s: len(s) == 1)
n_mixed = int((~ginfo.single_cond).sum())
print("groups:", len(ginfo), "| groups with conflicting conditions (excluded):", n_mixed)
ginfo = ginfo[ginfo.single_cond].copy()
ginfo["cond"] = ginfo.conds.map(lambda s: next(iter(s)))
ginfo["eval_ok"] = ginfo.any_eval_ok & (ginfo.all_facial if EVAL_FACIAL_ONLY else True)

split_of = {}
for cond, gg in ginfo.groupby("cond"):
    elig = sorted(gg[gg.eval_ok].index)
    rng.shuffle(elig)
    n_each = int(round(EVAL_FRACTION * len(elig)))
    n_each = max(min(MIN_EVAL_PHOTOS, int(len(elig) * MAX_EVAL_FRACTION)), n_each)
    n_each = min(n_each, int(len(elig) * MAX_EVAL_FRACTION)) if len(elig) else 0
    for gid in elig[:n_each]:
        split_of[gid] = "test"
    for gid in elig[n_each:2 * n_each]:
        split_of[gid] = "val"
    for gid in gg.index:
        split_of.setdefault(gid, "train")
    print("%-18s groups %5d | eval-eligible %5d | val %3d | test %3d | train %5d" % (
        cond, len(gg), len(elig), n_each, n_each, len(gg) - 2 * n_each))
m["split"] = m.duplicate_group.map(split_of)
m = m[m.split.notna()].copy()

def pick_eval_image(df):
    ok = df[df.decision.isin(EVAL_DECISIONS)]
    ok = ok[ok.facial_status == "facial"] if EVAL_FACIAL_ONLY else ok
    return ok.sort_values(["pixels", "n_kept_boxes", "image_id"], ascending=[False, False, True]).head(1)

ev = m[m.split.isin(["val", "test"])]
ev_keep = ev.groupby("duplicate_group", group_keys=False).apply(pick_eval_image)
tr = m[m.split == "train"]
tr = tr[tr.decision.isin(TRAIN_DECISIONS)]
if not TRAIN_NON_FACIAL:
    tr = tr[tr.facial_status == "facial"]
tr = tr.sort_values(["duplicate_group", "n_kept_boxes", "pixels"], ascending=[True, False, False]).groupby("duplicate_group").head(MAX_COPIES_PER_GROUP)
if MAX_TRAIN_PHOTOS_PER_CLASS:
    keep_g = set()
    for cond, d in tr.groupby("condition"):
        gs = sorted(d.duplicate_group.unique()); rng.shuffle(gs); keep_g |= set(gs[:MAX_TRAIN_PHOTOS_PER_CLASS])
    tr = tr[tr.duplicate_group.isin(keep_g)]
final = pd.concat([tr, ev_keep]).copy()

# safety: no photo group in more than one split
chk = final.groupby("duplicate_group").split.nunique()
assert (chk == 1).all(), "LEAK: a duplicate group appears in several splits"
print("\nfinal images per split and class (one image per photo in val/test):")
print(final.groupby(["split", "condition"]).size().unstack(fill_value=0).to_string())
print("no duplicate group crosses splits: OK")

## Stage 4 - Write the YOLO dataset + manifest
Copies images (originals are only read), writes YOLO label files and `data.yaml`, applies the oversampling of the weakest class to **training only**, and writes
`dataset_manifest.csv` with the split, source, condition, duplicate group and **box origin** of every image.

In [ ]:
if ds_dir.exists():
    shutil.rmtree(ds_dir)
for sp in ("train", "val", "test"):
    (ds_dir / sp / "images").mkdir(parents=True); (ds_dir / sp / "labels").mkdir(parents=True)

cid = {c: i for i, c in enumerate(CLASSES)}
ann_by = {k: v for k, v in ann.groupby("image_id")}
manifest, repeats_added = [], 0
from tqdm.auto import tqdm
for r in tqdm(list(final.itertuples()), total=len(final), desc="copying images"):
    ext = Path(r.original_filename).suffix.lower() or ".jpg"
    base = "%s_%s" % (r.image_id, r.duplicate_group)
    boxes = ann_by.get(r.image_id)
    lines = "" if boxes is None else "".join("%d %.6f %.6f %.6f %.6f\n" % (cid[b.cls], b.ncx, b.ncy, b.nw, b.nh) for b in boxes.itertuples())
    cls_name = COND_TO_CLASS[r.condition]
    times = OVERSAMPLE.get(cls_name, 1) if r.split == "train" else 1
    for k in range(times):
        nm = base if k == 0 else "%s_rep%d" % (base, k)
        shutil.copy2(r.source_path, ds_dir / r.split / "images" / (nm + ext))
        (ds_dir / r.split / "labels" / (nm + ".txt")).write_text(lines)
        if k:
            repeats_added += 1
    manifest.append({"image_id": r.image_id, "file": base + ext, "split": r.split, "condition": r.condition, "source_dataset": r.source_dataset,
                     "duplicate_group": r.duplicate_group, "facial_status": r.facial_status, "decision": r.decision, "n_boxes": r.n_kept_boxes,
                     "box_origin": "human-drawn (source dataset annotators)", "times_used_in_training": times})

yaml.safe_dump({"path": str(ds_dir.resolve()), "train": "train/images", "val": "val/images", "test": "test/images",
                "names": {i: c for i, c in enumerate(CLASSES)}}, open(ds_dir / "data.yaml", "w"))
man = pd.DataFrame(manifest)
man.to_csv(ds_dir / "dataset_manifest.csv", index=False)
print("dataset written:", ds_dir, "| oversampled copies added to train:", repeats_added)
print("manifest:", ds_dir / "dataset_manifest.csv")
DATA_YAML = ds_dir / "data.yaml"

## Stage 5 - Check the dataset (numbers + pictures) before training

In [ ]:
bx = []
for sp in ("train", "val", "test"):
    for lf in (ds_dir / sp / "labels").glob("*.txt"):
        for ln in lf.read_text().splitlines():
            t = ln.split()
            if len(t) == 5:
                bx.append({"split": sp, "class": CLASSES[int(t[0])], "area_pct": float(t[3]) * float(t[4]) * 100})
bx = pd.DataFrame(bx)
imgs = pd.DataFrame([{"split": sp, "n": len(list((ds_dir / sp / "images").glob("*")))} for sp in ("train", "val", "test")])
print("IMAGE FILES per split:"); print(imgs.to_string(index=False))
print("\nBOXES per split / class:"); print(bx.groupby(["split", "class"]).size().unstack(fill_value=0).to_string())
print("\nmedian box area (% of image) per class:"); print(bx.groupby("class").area_pct.median().round(3).to_string())
print("\nRule of thumb: >=1000 photos and >=3000 boxes per class for a solid model. Small classes (eczema) = pilot, report them separately.")

sample = man[man.split == "train"].sample(12, random_state=1)
fig, axs = plt.subplots(3, 4, figsize=(18, 13))
for ax, r in zip(axs.ravel(), sample.itertuples()):
    im = cv2.imdecode(np.fromfile(str(ds_dir / "train" / "images" / r.file), np.uint8), 1); H, W = im.shape[:2]
    for ln in (ds_dir / "train" / "labels" / (Path(r.file).stem + ".txt")).read_text().splitlines():
        c, x, y, w, h = ln.split(); c = int(c); x, y, w, h = float(x), float(y), float(w), float(h)
        col = [(0, 0, 255), (255, 0, 0), (0, 200, 255)][c]
        cv2.rectangle(im, (int((x - w / 2) * W), int((y - h / 2) * H)), (int((x + w / 2) * W), int((y + h / 2) * H)), col, max(2, W // 250))
    ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)); ax.set_title("%s | %s" % (r.condition, r.source_dataset[:30]), fontsize=8); ax.axis("off")
plt.tight_layout(); plt.show()

## Stage 6 - Train
Starts from COCO-pretrained weights. `yolov8n` is the default because the final system runs on a CPU-only Mini PC (see the benchmark in Stage 8).
With several `SEEDS` it trains once per seed so you can report the spread (important for the small eczema class).

In [ ]:
# --- free memory BEFORE training. DataLoader workers are copies of this process: the big audit tables above
#     would be duplicated in every worker and the machine runs out of RAM (this crashed a previous run).
import gc
for _n in ("master", "ann", "issues", "m", "final", "tr", "ev", "ev_keep", "ann_by", "ginfo", "withbox", "empty", "bx", "imgs", "sample", "dims", "nbox", "ok_box", "g"):
    globals().pop(_n, None)
gc.collect()
ds_dir = WORK / ("dataset_" + DATASET_VERSION)                   # (re)defined so training also works after a kernel restart
DATA_YAML = ds_dir / "data.yaml"
assert DATA_YAML.exists(), "dataset not built yet: run Stages 2-4 first"
man = pd.read_csv(ds_dir / "dataset_manifest.csv")

import torch
from ultralytics import YOLO

GPU = torch.cuda.is_available()
print("GPU:", torch.cuda.get_device_name(0) if GPU else "none (CPU training would take very long)")

RUNS = {}
for sd in SEEDS:
    name = "detector_%s_%s_seed%d" % (DATASET_VERSION, Path(MODEL).stem, sd)
    mdl = YOLO(MODEL)
    mdl.train(data=str(DATA_YAML), imgsz=IMGSZ, epochs=EPOCHS, patience=PATIENCE, batch=BATCH if GPU else 4, device=0 if GPU else "cpu",
              workers=WORKERS, seed=sd, cos_lr=True, close_mosaic=10,
              project=str((WORK / "runs").resolve()), name=name, exist_ok=True,
              fliplr=0.5, flipud=0.0, degrees=10, scale=0.5, hsv_h=0.015, hsv_s=0.5, hsv_v=0.35)
    RUNS[sd] = Path(mdl.trainer.save_dir) / "weights" / "best.pt"
    print("seed", sd, "->", RUNS[sd])

## Stage 7 - Evaluate on the held-out TEST split  (these are the numbers for the paper)

In [ ]:
results = {}
for sd, best in RUNS.items():
    met = YOLO(str(best)).val(data=str(DATA_YAML), split="test", imgsz=IMGSZ, conf=0.001, plots=True)
    per = {}
    for k, ci in enumerate(met.box.ap_class_index):
        p, r, ap50, ap = met.box.class_result(k)
        per[CLASSES[int(ci)]] = {"precision": float(p), "recall": float(r), "mAP50": float(ap50), "mAP50_95": float(ap)}
    results[sd] = {"overall": {"precision": float(met.box.mp), "recall": float(met.box.mr), "mAP50": float(met.box.map50), "mAP50_95": float(met.box.map)}, "per_class": per}

rows = []
for sd, r in results.items():
    for cname, v in r["per_class"].items():
        rows.append(dict(seed=sd, cls=cname, **v))
tab = pd.DataFrame(rows)
print("TEST-SET RESULTS per class and seed"); print(tab.round(3).to_string(index=False))
if len(SEEDS) > 1:
    print("\nmean +- std over seeds:")
    print(tab.groupby("cls")[["precision", "recall", "mAP50", "mAP50_95"]].agg(["mean", "std"]).round(3).to_string())
print("\nReport eczema separately: it is the weakest class (smallest training set).")
test_n = pd.read_csv(ds_dir / "dataset_manifest.csv").query("split == 'test'").groupby("condition").size()
print("\ntest photos per class:", test_n.to_dict(), "(small test sets = wide uncertainty; say so)")

# visual check on test images
BEST = RUNS[SEEDS[0]]
tfiles = sorted((ds_dir / "test" / "images").glob("*"))
random.Random(0).shuffle(tfiles)
res = YOLO(str(BEST)).predict([str(p) for p in tfiles[:12]], conf=CONF, imgsz=IMGSZ, verbose=False)
fig, axs = plt.subplots(3, 4, figsize=(18, 13))
for ax, r in zip(axs.ravel(), res):
    ax.imshow(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB)); ax.axis("off")
plt.tight_layout(); plt.show()

## Stage 8 - Export ONNX and benchmark on CPU
The deployment machine is a CPU-only Mini PC (4 cores). The benchmark uses 4 threads. Run the same cell on the Mini PC for the real number.

In [ ]:
import onnxruntime as ort

onnx_src = Path(YOLO(str(BEST)).export(format="onnx", imgsz=IMGSZ, opset=12, simplify=True, dynamic=False))
EXP = WORK / "export"; EXP.mkdir(parents=True, exist_ok=True)
onnx_path = EXP / ("lesion_detector_%s.onnx" % DATASET_VERSION)
shutil.copy2(onnx_src, onnx_path)

so = ort.SessionOptions(); so.intra_op_num_threads = 4
sess = ort.InferenceSession(str(onnx_path), so, providers=["CPUExecutionProvider"])
inp = sess.get_inputs()[0]
x = np.random.rand(1, 3, IMGSZ, IMGSZ).astype("float32")
for _ in range(3):
    sess.run(None, {inp.name: x})
ts = []
for _ in range(20):
    t0 = time.perf_counter(); out = sess.run(None, {inp.name: x}); ts.append((time.perf_counter() - t0) * 1000)
lat = {"median_ms": round(float(np.median(ts)), 1), "p95_ms": round(float(np.percentile(ts, 95)), 1), "threads": 4}
print("ONNX ok | input", inp.name, inp.shape, "| output", out[0].shape, "| CPU latency (this machine):", lat)

meta = {
    "model": "PrecisionSkin lesion detector", "weights": MODEL, "dataset_version": DATASET_VERSION,
    "created": datetime.datetime.now().isoformat(timespec="seconds"),
    "classes": CLASSES, "imgsz": IMGSZ, "default_conf": CONF, "nms_iou": 0.45,
    "onnx_sha256": hashlib.sha256(onnx_path.read_bytes()).hexdigest(),
    "test_metrics_by_seed": {str(k): v for k, v in results.items()},
    "split_images": man.groupby("split").size().to_dict(),
    "test_photos_per_class": test_n.to_dict(),
    "box_origin": "human-drawn by the source dataset annotators; no machine-generated boxes",
    "cpu_latency": lat,
    "preprocess": "letterbox to IMGSZxIMGSZ (pad 114), RGB, float32 / 255, NCHW",
    "output": "[1, 4+num_classes, N]: rows cx,cy,w,h (pixels of the letterboxed input) then class scores; no objectness; apply NMS",
    "note": "Research prototype. Boxes show candidate lesion locations; not a diagnosis. Eczema is a pilot class.",
}
json.dump(meta, open(EXP / ("lesion_detector_%s.json" % DATASET_VERSION), "w"), indent=2, default=str)
print("saved:", onnx_path); print("saved:", EXP / ("lesion_detector_%s.json" % DATASET_VERSION))